## University of San Antonio Social Impact AI Datathon

Use this notebook to set up your working environment and start your project.


### Getting started: Dell Pro Max GB10 
The NVIDIA Sync App is the easiest way to work with a GB10. Download and user docs for Sync: https://docs.nvidia.com/dgx/dgx-spark/nvidia-sync.html
 
Install Sync on your laptop and follow these steps to connect to the GB10 at your workstation: https://docs.nvidia.com/sync/latest/getting-started.html

Make sure to include VSCode if you want to work from a notebook or editor. 

Once Sync is installed, open it and click **VSCode** to launch the editor, or **Terminal** if you prefer to use command line/ssh.


#### Using Terminal/ssh/bash

- If you're running code through a jupyter notebook code chunk, you'll need to include an exclamation point ! before each bash command. 
```
!docker ps
```

- If you're running bash code directly in a terminal or command line interface (CLI), be sure to remove the leading !
```
docker ps
```

- If you want to run python scripts directly in terminal, be sure to include the path and ```python``` at the beginning of the command. You can also run python scripts directly from github using the file path. 


#### Virtual Environments and Containers
A virtual environment is an isolated Python workspace on the GB10 that keeps each project's packages and versions separate, so installing vLLM or PyTorch for one project doesn't break another. 

This matters on the GB10 because its Arm64 (aarch64) architecture and CUDA setup make package versions especially sensitive, and a clean environment, or a container like NVIDIA's PyTorch image, makes problems easier to reproduce and fix. 

Docker packages an application together with its entire software environment (the operating system libraries, CUDA and driver-facing tools, Python, and all dependencies) into a container that runs the same way anywhere. A Python virtual environment isolates only Python packages and still uses the host's system libraries, while a Docker container isolates the whole user-space stack, which makes it more reproducible but heavier. 


#### Running commands in terminal
If you encounter errors related to permissions or access, try using ```sudo``` before your command.

#### Pre-loaded GB10 environment

The GB10 devices have been pre-loaded with some resources to save time during the event. Here's what we installed:
- Docker
- HuggingFace command line interface
- vLLM
- Qwen3-VL-8B-Instruct model


#### Working with HuggingFace

If you don't have a HuggingFace account, go ahead and create one. Make a note of your username and password. 

Run the next command to log in to HuggingFace. This will print a URL and a short code. 

Copy the url and open it in a web browser on your device to log in with your hf credentials. 

Enter the code, approve the request, and the CLI retrieves and saves an access token on your machine.

In [ ]:
# log in to huggingface (bash)
env PATH="/home/dell/.local/bin:$PATH" hf auth login

# copy the URL that was generated and open it in a browser to log in

#### Downloading files from HuggingFace

``` hf download --help ```

To download a single file from a repo, simply provide the repo_id and filename:
The example below pulls the 'config.json' file from the 'gpt2' repo
``` hf download gpt2 config.json ```

To download an entire repo, specify the repo ID:
``` hf download HuggingFaceH4/zephyr-7b-beta ```

``--dry-run`` returns a list of all repo files, file size, and download status:
``` hf download openai-community/gpt2 --dry-run ```

To download a dataset, a Space or a kernel, use the ``--repo-type`` option:

Example - https://huggingface.co/datasets/HuggingFaceH4/ultrachat_200k
``` hf download HuggingFaceH4/ultrachat_200k --repo-type dataset ```

To delete a model from your local device, use ```hf cache list``` to identify cached models. Remove one like this:
``` hf cache rm <paste_model_name>```

To remove docker images, first run ```docker images``` to get the list of all installed images. Remove one like this:
``` docker rmi <image_name_or_id> ```

### AI Models & Developer Tools

**Base AI models** are trained only to predict the next token on large amounts of raw text. It is good at continuing text, but it doesn't reliably follow instructions or answer questions directly. It may ramble, repeat itself, or continue your prompt instead of responding to it.

**Instruct** or **chat** model versions start from the base model and get further training, usually supervised fine-tuning on instruction and response examples, often followed by preference tuning. That training teaches it to follow directions, hold a conversation in a chat template, and respond in a helpful, safe format.

Instruct models are usually the easier choice for small, task-specific datasets, because they already know how to follow instructions. We recommend that you choose a multi-modal instruct model for today's event. 

**Multi-modal models** are AI models that can understand and work with more than one type of data, such as text, images, audio, and video, within a single system. This lets them do tasks like describing a photo, answering questions about a chart, or transcribing and summarizing a video, instead of handling only written language.

This link should show a filtered list of models that would be ideal for the GB10 and for today's event (filtered for Instruct, between 6B 200B parameters, vLLM optimized):
https://huggingface.co/models?num_parameters=min:6B,max:128B&apps=vllm&inference_provider=all&sort=downloads&search=instruct

Choose a model based on the task: Image-to-text, Visual Question Answering, or Any-to-Any

**Other Models:**
- https://huggingface.co/Qwen/Qwen3-VL-30B-A3B-Instruct 
- https://huggingface.co/DavidAU/Qwen3.5-27B-Claude-4.6-OS-INSTRUCT 
- https://recipes.vllm.ai/mistralai/Ministral-3-14B-Instruct-2512 
- https://recipes.vllm.ai/nvidia/NVIDIA-Nemotron-3.5-Super-VL-120B-A12B-NVFP4?hardware=dgx_spark_gb10&fmode=spec_decoding%3Adspark 
- https://recipes.vllm.ai/mistralai/Mistral-Small-4-119B-2603 
 


In [ ]:
# Copy a repository to your machine
git clone <repo-url>

# Stage + commit your changes
git add .
git commit -m "describe what you changed"

# Share with the team / get their changes
git push
git pull

# Work on a separate branch
git checkout -b my-feature

#### vLLM on GB10

vLLM is an open-source engine for serving large language models (LLMs) quickly and efficiently. You load a model into it, and it exposes an API (OpenAI-compatible) that your apps, scripts, or chat UIs can call. It's used for inference, meaning running the model, not for training it.

A typical vLLM server answers at an address like http://localhost:8000/v1, and you can check which models it's serving at /v1/models. 

vLLM and one model are already installed on your GB10 device. Use the code below to launch vLLM and serve the model. 

In [ ]:
# launch the vLLM container and serve the model

docker run -d \
  --gpus all \
  -p 8000:8000 \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  --ipc=host \
  --name qwen-vl-server \
  c1c9f6fd5c10 \
  --model Qwen/Qwen3-VL-8B-Instruct \
  --host 0.0.0.0 \
  --port 8000 \
  --gpu-memory-utilization 0.90 \
  --max-model-len 8192

docker logs -f qwen-vl-server

